In [5]:
import os
import sys
import json
import time
from dataclasses import dataclass, asdict
from typing import Any, Literal

# Notebook imports: mirror style used in other notebooks
current_dir = os.getcwd()
functions_path = os.path.join(current_dir, "Functions")
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

from Functions.utils.imports import *
from Functions.data_loading import load_event_log, LOG_PATHS
from Functions.data_loading.pm4py_helpers import assign_tau_labels
from Functions.tree_conversion import tree_to_named_pattern_expression
from Functions.logical_spec import WorkflowPatternTemplate
from Functions.properties import extract_ini_fin, build_full_spec, evaluate_property
from Functions.shapley import shapley_mc_permutations, shapley_random_subsets
from Functions.players.player_enumeration import list_players_from_expression
from Functions.process_tree import count_nodes

import pm4py
from pm4py.algo.discovery.inductive import algorithm as inductive_algo

# ---------- paths / cache ----------
RESULTS_DIR = os.path.join("..", "Docs", "Problems", "shapley_values", "experiments", "Hierarchic")
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_JSON = os.path.join(RESULTS_DIR, "hierarchic_results.json")

TEMPLATES = WorkflowPatternTemplate.load_pattern_property_set("../Data/patterns.json")

PropertyName = Literal["satisfiability", "liveness", "safety"]
VariantName = Literal["IM", "IMf", "IMd", "HM"]

@dataclass(frozen=True)
class ShapleyCfg:
    method: Literal["mc", "rs"] = "mc"
    n_perm: int = 400
    n_samples: int = 800
    seed: int = 2025
    keep_frac_range: tuple[float, float] = (0.5, 0.8)


@dataclass(frozen=True)
class ExperimentCfg:
    log_name: str
    variant: VariantName
    noise_threshold: float
    # heuristics miner (proxy for parallelism threshold)
    hm_and_threshold: float | None = None
    # dfg denoising
    min_directly_follows_freq: int | None = None
    # shapley-guided pruning
    prune_eps: float | None = None


def _phi_buckets(phi: dict[str, float], *, critical_abs: float = 0.1, neutral_abs: float = 0.01):
    harmful = sum(1 for v in phi.values() if v < 0)
    neutral = sum(1 for v in phi.values() if abs(v) < neutral_abs)
    critical = sum(1 for v in phi.values() if abs(v) >= critical_abs)
    avg_abs = sum(abs(v) for v in phi.values()) / max(1, len(phi))
    top5 = sorted(phi.items(), key=lambda kv: kv[1], reverse=True)[:5]
    return harmful, neutral, critical, avg_abs, top5


def _discover_tree_inductive_variant(log, *, variant: VariantName, noise_threshold: float):
    if variant == "HM":
        raise ValueError("HM is handled in a separate discovery function")

    var_map = {
        "IM": inductive_algo.Variants.IM,
        "IMf": inductive_algo.Variants.IMf,
        "IMd": inductive_algo.Variants.IMd,
    }

    # IMf reads noise_threshold from parameters (string key is accepted by exec_utils)
    params = {"noise_threshold": float(noise_threshold)}
    tree = inductive_algo.apply(log, parameters=params, variant=var_map[variant])
    return assign_tau_labels(tree)


def _discover_tree_heuristics_as_tree(log, *, and_threshold: float):
    """Mine a process tree using Heuristics Miner as a proxy for parallelism tuning.

    Note: the resulting Petri net is not guaranteed to be a WF-net, so conversion to a
    process tree can fail. We surface this as a RuntimeError so the experiment runner
    can skip/record the failure without killing the notebook.
    """
    net, im, fm = pm4py.discover_petri_net_heuristics(log, and_threshold=and_threshold)
    try:
        tree = pm4py.convert_to_process_tree(net, im, fm)
    except Exception as exc:
        raise RuntimeError(f"HM->ProcessTree conversion failed (and_threshold={and_threshold}): {exc}") from exc
    return assign_tau_labels(tree)


def _filter_dfg_by_min_freq(
    dfg_graph: dict[tuple[str, str], int],
    start_acts: dict[str, int],
    end_acts: dict[str, int],
    *,
    min_freq: int,
):
    from pm4py.objects.dfg.obj import DFG

    kept_edges = {e: f for e, f in dfg_graph.items() if f >= min_freq}

    acts_in_edges = set()
    for (a, b) in kept_edges.keys():
        acts_in_edges.add(a)
        acts_in_edges.add(b)

    kept_start = {a: f for a, f in start_acts.items() if a in acts_in_edges}
    kept_end = {a: f for a, f in end_acts.items() if a in acts_in_edges}

    dfg = DFG()
    for k, v in kept_edges.items():
        dfg.graph[k] = v
    for a, f in kept_start.items():
        dfg.start_activities[a] = f
    for a, f in kept_end.items():
        dfg.end_activities[a] = f
    return dfg


def _discover_tree_from_dfg_denoised(log, *, min_directly_follows_freq: int):
    dfg_graph, start_acts, end_acts = pm4py.discover_dfg(log)
    dfg = _filter_dfg_by_min_freq(dfg_graph, start_acts, end_acts, min_freq=min_directly_follows_freq)
    tree = inductive_algo.apply(dfg, variant=inductive_algo.Variants.IMd)
    return assign_tau_labels(tree)


def _compute_properties(named_expr: str) -> tuple[int, int, int]:
    spec = build_full_spec(named_expr, TEMPLATES)
    ini, fin = extract_ini_fin(named_expr, TEMPLATES)
    sat = evaluate_property(spec, "satisfiability")
    liv = evaluate_property(spec, "liveness", ini=ini, fin=fin)
    saf = evaluate_property(spec, "safety", ini=ini, fin=fin)
    return sat, liv, saf


def _compute_shapley(named_expr: str, prop: PropertyName, cfg: ShapleyCfg) -> tuple[dict[str, float], float]:
    t0 = time.time()
    if cfg.method == "mc":
        phi, _meta = shapley_mc_permutations(
            named_expr,
            TEMPLATES,
            prop,
            n_perm=cfg.n_perm,
            seed=cfg.seed,
            progress_every=max(25, cfg.n_perm // 8),
        )
    else:
        phi, _meta = shapley_random_subsets(
            named_expr,
            TEMPLATES,
            prop,
            n_samples=cfg.n_samples,
            keep_frac_range=cfg.keep_frac_range,
            seed=cfg.seed,
            progress_every=max(100, cfg.n_samples // 8),
        )
    return phi, (time.time() - t0)


def _load_results() -> list[dict[str, Any]]:
    if not os.path.exists(RESULTS_JSON):
        return []
    with open(RESULTS_JSON, "r", encoding="utf-8") as f:
        return json.load(f)


def _save_results(rows: list[dict[str, Any]]):
    with open(RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)


# ---------- configuration ----------

PROPERTIES: list[PropertyName] = ["satisfiability", "liveness", "safety"]

SHAPLEY_CFG = ShapleyCfg(method="mc", n_perm=300, seed=2025)

CRITICAL_ABS = 0.10
NEUTRAL_ABS = 0.01

A_VARIANTS: list[VariantName] = ["IM", "IMf", "IMd"]
A_NOISE: list[float] = [0.0, 0.1, 0.2, 0.3]

B_AND_THRESHOLDS: list[float] = [0.3, 0.5, 0.7]

C_MIN_DF_FREQS: list[int] = [1, 3, 5]
C_PRUNE_EPS: float = 0.01

LOGS_TO_RUN = ["running_example"]  # change to list(LOG_PATHS.keys()) for full sweep

print("Ready.")
print("Results cache:", RESULTS_JSON)
print("Logs available:", list(LOG_PATHS.keys()))
print("Running:", LOGS_TO_RUN)



Ready.
Results cache: ../Docs/Problems/shapley_values/experiments/Hierarchic/hierarchic_results.json
Logs available: ['running_example', 'hospital_billing', 'bpi_2012']
Running: ['running_example']


In [6]:
# ---------- experiment runner ----------

def _run_one_config(cfg: ExperimentCfg, *, shapley_cfg: ShapleyCfg) -> dict[str, Any]:
    """Run a single configuration.

    Never throws: on failure returns a row with an `error` field.
    """
    try:
        log = load_event_log(LOG_PATHS[cfg.log_name])

        # 1) Mine tree
        t0 = time.time()
        if cfg.variant == "HM":
            assert cfg.hm_and_threshold is not None
            tree = _discover_tree_heuristics_as_tree(log, and_threshold=cfg.hm_and_threshold)
        elif cfg.min_directly_follows_freq is not None:
            tree = _discover_tree_from_dfg_denoised(log, min_directly_follows_freq=cfg.min_directly_follows_freq)
        else:
            tree = _discover_tree_inductive_variant(log, variant=cfg.variant, noise_threshold=cfg.noise_threshold)
        mining_seconds = time.time() - t0

        nodes = count_nodes(tree)

        # 2) Convert to named expression
        named_expr = tree_to_named_pattern_expression(tree)

        # 3) Properties
        sat, liv, saf = _compute_properties(named_expr)

        # 4) Shapley for each property
        per_prop: dict[str, Any] = {}
        for prop in PROPERTIES:
            phi, phi_seconds = _compute_shapley(named_expr, prop, shapley_cfg)

            # optional pruning experiment: for now we only report what would be pruned
            prunable = [pid for pid, v in phi.items() if cfg.prune_eps is not None and abs(v) < cfg.prune_eps]

            harmful, neutral, critical, avg_abs, top5 = _phi_buckets(
                phi, critical_abs=CRITICAL_ABS, neutral_abs=NEUTRAL_ABS
            )

            per_prop[prop] = {
                "shapley": phi,
                "shapley_seconds": phi_seconds,
                "harmful": harmful,
                "neutral": neutral,
                "critical": critical,
                "avg_abs_phi": avg_abs,
                "top5": top5,
                "prunable_count": len(prunable),
            }

        players = len(list_players_from_expression(named_expr))

        return {
            "cfg": asdict(cfg),
            "error": None,
            "mining_seconds": mining_seconds,
            "nodes": nodes,
            "players": players,
            "properties": {"sat": sat, "liv": liv, "saf": saf},
            "per_property": per_prop,
        }

    except Exception as exc:
        return {
            "cfg": asdict(cfg),
            "error": str(exc),
        }


def run_all(*, force: bool = False) -> list[dict[str, Any]]:
    existing = _load_results()
    if existing and not force:
        print(f"[CACHE] Using cached results: {len(existing)} rows -> {RESULTS_JSON}")
        return existing

    rows: list[dict[str, Any]] = []

    # --- A: IM variant × noise ---
    for log_name in LOGS_TO_RUN:
        for variant in A_VARIANTS:
            for noise in A_NOISE:
                cfg = ExperimentCfg(log_name=log_name, variant=variant, noise_threshold=noise)
                print(f"[A] log={log_name} variant={variant} noise={noise}")
                rows.append(_run_one_config(cfg, shapley_cfg=SHAPLEY_CFG))

    # --- B: parallelism threshold proxy (Heuristics Miner) ---
    # HM -> PetriNet -> ProcessTree conversion may fail (not a WF-net). We record the error and continue.
    for log_name in LOGS_TO_RUN:
        for thr in B_AND_THRESHOLDS:
            cfg = ExperimentCfg(log_name=log_name, variant="HM", noise_threshold=0.0, hm_and_threshold=thr)
            print(f"[B] log={log_name} HM and_threshold={thr}")
            r = _run_one_config(cfg, shapley_cfg=SHAPLEY_CFG)
            if r.get("error"):
                print(f"      [B] skipped: {r['error']}")
            rows.append(r)

    # --- C: DFG denoising + report prunable nodes ---
    for log_name in LOGS_TO_RUN:
        for min_df in C_MIN_DF_FREQS:
            cfg = ExperimentCfg(
                log_name=log_name,
                variant="IMd",
                noise_threshold=0.0,
                min_directly_follows_freq=min_df,
                prune_eps=C_PRUNE_EPS,
            )
            print(f"[C] log={log_name} min_DF_freq={min_df} prune_eps={C_PRUNE_EPS}")
            rows.append(_run_one_config(cfg, shapley_cfg=SHAPLEY_CFG))

    _save_results(rows)
    print(f"[CACHE] Saved {len(rows)} rows -> {RESULTS_JSON}")
    return rows



In [7]:
# ---------- run ----------

results = run_all(force=False)
print('Rows:', len(results))

# quick peek
print(json.dumps(results[0]["cfg"], ensure_ascii=False, indent=2))
print(results[0]["properties"])



[A] log=running_example variant=IM noise=0.0


parsing log, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

      ... MC progress 37/300 (players=6, cached=56) [0.0s]
      ... MC progress 74/300 (players=6, cached=64) [0.0s]
      ... MC progress 111/300 (players=6, cached=64) [0.0s]
      ... MC progress 148/300 (players=6, cached=64) [0.0s]
      ... MC progress 185/300 (players=6, cached=64) [0.0s]
      ... MC progress 222/300 (players=6, cached=64) [0.0s]
      ... MC progress 259/300 (players=6, cached=64) [0.0s]
      ... MC progress 296/300 (players=6, cached=64) [0.0s]
      ... MC progress 37/300 (players=6, cached=56) [0.0s]
      ... MC progress 74/300 (players=6, cached=64) [0.0s]
      ... MC progress 111/300 (players=6, cached=64) [0.0s]
      ... MC progress 148/300 (players=6, cached=64) [0.0s]
      ... MC progress 185/300 (players=6, cached=64) [0.0s]
      ... MC progress 222/300 (players=6, cached=64) [0.0s]
      ... MC progress 259/300 (players=6, cached=64) [0.0s]
      ... MC progress 296/300 (players=6, cached=64) [0.0s]
      ... MC progress 37/300 (players=6, cac

parsing log, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

      ... MC progress 37/300 (players=6, cached=56) [0.0s]
      ... MC progress 74/300 (players=6, cached=64) [0.0s]
      ... MC progress 111/300 (players=6, cached=64) [0.0s]
      ... MC progress 148/300 (players=6, cached=64) [0.0s]
      ... MC progress 185/300 (players=6, cached=64) [0.0s]
      ... MC progress 222/300 (players=6, cached=64) [0.0s]
      ... MC progress 259/300 (players=6, cached=64) [0.0s]
      ... MC progress 296/300 (players=6, cached=64) [0.0s]
      ... MC progress 37/300 (players=6, cached=56) [0.0s]
      ... MC progress 74/300 (players=6, cached=64) [0.0s]
      ... MC progress 111/300 (players=6, cached=64) [0.0s]
      ... MC progress 148/300 (players=6, cached=64) [0.0s]
      ... MC progress 185/300 (players=6, cached=64) [0.0s]
      ... MC progress 222/300 (players=6, cached=64) [0.0s]
      ... MC progress 259/300 (players=6, cached=64) [0.0s]
      ... MC progress 296/300 (players=6, cached=64) [0.0s]
      ... MC progress 37/300 (players=6, cac

...

      ... MC progress 37/300 (players=6, cached=56) [0.0s]
      ... MC progress 74/300 (players=6, cached=64) [0.0s]
      ... MC progress 111/300 (players=6, cached=64) [0.0s]
      ... MC progress 148/300 (players=6, cached=64) [0.0s]
      ... MC progress 185/300 (players=6, cached=64) [0.0s]
      ... MC progress 222/300 (players=6, cached=64) [0.0s]
      ... MC progress 259/300 (players=6, cached=64) [0.0s]
      ... MC progress 296/300 (players=6, cached=64) [0.0s]
      ... MC progress 37/300 (players=6, cached=56) [0.0s]
      ... MC progress 74/300 (players=6, cached=64) [0.0s]
      ... MC progress 111/300 (players=6, cached=64) [0.0s]
      ... MC progress 148/300 (players=6, cached=64) [0.0s]
      ... MC progress 185/300 (players=6, cached=64) [0.0s]
      ... MC progress 222/300 (players=6, cached=64) [0.0s]
      ... MC progress 259/300 (players=6, cached=64) [0.0s]
      ... MC progress 296/300 (players=6, cached=64) [0.0s]
      ... MC progress 37/300 (players=6, cac

parsing log, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

      ... MC progress 37/300 (players=2, cached=4) [0.0s]
      ... MC progress 74/300 (players=2, cached=4) [0.0s]
      ... MC progress 111/300 (players=2, cached=4) [0.0s]
      ... MC progress 148/300 (players=2, cached=4) [0.0s]
      ... MC progress 185/300 (players=2, cached=4) [0.0s]
      ... MC progress 222/300 (players=2, cached=4) [0.0s]
      ... MC progress 259/300 (players=2, cached=4) [0.0s]
      ... MC progress 296/300 (players=2, cached=4) [0.0s]
      ... MC progress 37/300 (players=2, cached=4) [0.0s]
      ... MC progress 74/300 (players=2, cached=4) [0.0s]
      ... MC progress 111/300 (players=2, cached=4) [0.0s]
      ... MC progress 148/300 (players=2, cached=4) [0.0s]
      ... MC progress 185/300 (players=2, cached=4) [0.0s]
      ... MC progress 222/300 (players=2, cached=4) [0.0s]
      ... MC progress 259/300 (players=2, cached=4) [0.0s]
      ... MC progress 296/300 (players=2, cached=4) [0.0s]
      ... MC progress 37/300 (players=2, cached=4) [0.0s]
  

parsing log, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

[CACHE] Saved 18 rows -> ../Docs/Problems/shapley_values/experiments/Hierarchic/hierarchic_results.json
Rows: 18
{
  "log_name": "running_example",
  "variant": "IM",
  "noise_threshold": 0.0,
  "hm_and_threshold": null,
  "min_directly_follows_freq": null,
  "prune_eps": null
}
{'sat': 1, 'liv': 1, 'saf': 0}


In [8]:
# ---------- report helpers ----------

import pandas as pd


def _flatten(results: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in results:
        base = {
            **r["cfg"],
            "nodes": r["nodes"],
            "players": r["players"],
            "mining_seconds": r["mining_seconds"],
            "sat": r["properties"]["sat"],
            "liv": r["properties"]["liv"],
            "saf": r["properties"]["saf"],
        }
        for prop, pr in r["per_property"].items():
            rows.append({
                **base,
                "property": prop,
                "shapley_seconds": pr["shapley_seconds"],
                "harmful": pr["harmful"],
                "neutral": pr["neutral"],
                "critical": pr["critical"],
                "avg_abs_phi": pr["avg_abs_phi"],
                "prunable_count": pr.get("prunable_count", 0),
                "top5": pr["top5"],
            })
    return pd.DataFrame(rows)


df = _flatten([r for r in results if not r.get("error")])

# show failures (if any)
failed = [r for r in results if r.get("error")]
if failed:
    print(f"Failed configs: {len(failed)}")
    for r in failed[:10]:
        print(r["cfg"], "->", r["error"])

df.head()


Failed configs: 4
{'log_name': 'running_example', 'variant': 'HM', 'noise_threshold': 0.0, 'hm_and_threshold': 0.3, 'min_directly_follows_freq': None, 'prune_eps': None} -> HM->ProcessTree conversion failed (and_threshold=0.3): Parsing of WF-net Failed
{'log_name': 'running_example', 'variant': 'HM', 'noise_threshold': 0.0, 'hm_and_threshold': 0.5, 'min_directly_follows_freq': None, 'prune_eps': None} -> HM->ProcessTree conversion failed (and_threshold=0.5): Parsing of WF-net Failed
{'log_name': 'running_example', 'variant': 'HM', 'noise_threshold': 0.0, 'hm_and_threshold': 0.7, 'min_directly_follows_freq': None, 'prune_eps': None} -> HM->ProcessTree conversion failed (and_threshold=0.7): Parsing of WF-net Failed
{'log_name': 'running_example', 'variant': 'IMd', 'noise_threshold': 0.0, 'hm_and_threshold': None, 'min_directly_follows_freq': 5, 'prune_eps': 0.01} -> list index out of range


,log_name,variant,noise_threshold,hm_and_threshold,min_directly_follows_freq,prune_eps,nodes,players,mining_seconds,sat,liv,saf,property,shapley_seconds,harmful,neutral,critical,avg_abs_phi,prunable_count,top5
0,running_example,IM,0.0,None,NaN,NaN,14,6,0.005874,1,1,0,satisfiability,0.032178,1,2,3,0.151111,0,"[(Seq2@2@2, 0.24), (Seq2@3, 0.1333333333333333..."
1,running_example,IM,0.0,None,NaN,NaN,14,6,0.005874,1,1,0,liveness,0.028551,0,6,0,0.000000,0,"[(Seq2@5, 0.0), (Seq2@4, 0.0), (Seq2@3, 0.0), ..."
2,running_example,IM,0.0,None,NaN,NaN,14,6,0.005874,1,1,0,safety,0.016568,0,6,0,0.000000,0,"[(Seq2@5, 0.0), (Seq2@4, 0.0), (Seq2@3, 0.0), ..."
3,running_example,IM,0.1,None,NaN,NaN,14,6,0.002955,1,1,0,satisfiability,0.016940,1,2,3,0.151111,0,"[(Seq2@2@2, 0.24), (Seq2@3, 0.1333333333333333..."
4,running_example,IM,0.1,None,NaN,NaN,14,6,0.002955,1,1,0,liveness,0.017383,0,6,0,0.000000,0,"[(Seq2@5, 0.0), (Seq2@4, 0.0), (Seq2@3, 0.0), ..."


In [9]:
# (moved) Setup + definitions are in the first code cell now.
# This cell is intentionally left empty to keep execution order clean.



In [10]:
# ---------- summaries ----------

# A) variant × noise: choose baseline config with minimal harmful and stable top-5

def _pick_baseline(df: pd.DataFrame) -> pd.DataFrame:
    a = df[df["variant"].isin(["IM", "IMf", "IMd"]) & df["min_directly_follows_freq"].isna() & df["hm_and_threshold"].isna()].copy()
    # aggregate across properties
    g = a.groupby(["log_name", "variant", "noise_threshold"], dropna=False).agg(
        harmful_mean=("harmful", "mean"),
        neutral_mean=("neutral", "mean"),
        critical_mean=("critical", "mean"),
        nodes_mean=("nodes", "mean"),
        mining_seconds_mean=("mining_seconds", "mean"),
        sat_min=("sat", "min"),
        liv_min=("liv", "min"),
        saf_min=("saf", "min"),
    ).reset_index()
    return g.sort_values(["sat_min", "liv_min", "saf_min", "harmful_mean", "nodes_mean"], ascending=[False, False, False, True, True])

baseline_rank = _pick_baseline(df)
baseline_rank.head(20)


,log_name,variant,noise_threshold,harmful_mean,neutral_mean,critical_mean,nodes_mean,mining_seconds_mean,sat_min,liv_min,saf_min
0,running_example,IM,0.0,0.333333,4.666667,1.0,14.0,0.005874,1,1,0
1,running_example,IM,0.1,0.333333,4.666667,1.0,14.0,0.002955,1,1,0
2,running_example,IM,0.2,0.333333,4.666667,1.0,14.0,0.002437,1,1,0
3,running_example,IM,0.3,0.333333,4.666667,1.0,14.0,0.003576,1,1,0
4,running_example,IMd,0.0,0.333333,4.666667,1.0,14.0,0.004967,1,1,0
5,running_example,IMd,0.1,0.333333,4.666667,1.0,14.0,0.002280,1,1,0
6,running_example,IMd,0.2,0.333333,4.666667,1.0,14.0,0.001752,1,1,0
7,running_example,IMd,0.3,0.333333,4.666667,1.0,14.0,0.001643,1,1,0
8,running_example,IMf,0.0,0.333333,4.666667,1.0,14.0,0.002908,1,1,0
9,running_example,IMf,0.1,0.333333,4.666667,1.0,14.0,0.001974,1,1,0


In [11]:
# B) heuristics miner and_threshold impact (proxy for parallelism threshold)

b = df[df["variant"].eq("HM")].copy()
if not b.empty:
    b_summary = b.groupby(["log_name", "hm_and_threshold", "property"], dropna=False).agg(
        harmful_mean=("harmful", "mean"),
        avg_abs_phi_mean=("avg_abs_phi", "mean"),
        nodes_mean=("nodes", "mean"),
        liv_min=("liv", "min"),
        sat_min=("sat", "min"),
        saf_min=("saf", "min"),
    ).reset_index().sort_values(["log_name", "hm_and_threshold", "property"])
    b_summary
else:
    print('No B results')



No B results


In [12]:
# C) DFG denoising + Shapley-guided pruning report

c = df[df["min_directly_follows_freq"].notna()].copy()
if not c.empty:
    c_summary = c.groupby(["log_name", "min_directly_follows_freq", "property"], dropna=False).agg(
        nodes_mean=("nodes", "mean"),
        harmful_mean=("harmful", "mean"),
        neutral_mean=("neutral", "mean"),
        critical_mean=("critical", "mean"),
        prunable_mean=("prunable_count", "mean"),
        sat_min=("sat", "min"),
        liv_min=("liv", "min"),
        saf_min=("saf", "min"),
    ).reset_index().sort_values(["log_name", "min_directly_follows_freq", "property"])
    c_summary
else:
    print('No C results')

